# Ada — train on a real dataset (Colab)

Tokenizers are trained **unsupervised** — they need raw text, not labels. This notebook streams
a 2026-frontier corpus and trains **both**:
1. **Ada v0** (statistical superword+entropy BPE) on real text, and re-runs the ablation so we
   finally see whether the BLT entropy-guard helps on *natural* text (it was only marginal on
   synthetic text — FINDINGS §5).
2. **Ada v1** (neural: H-Net router + FSQ bottleneck + MrT5 delete gate) on the same bytes.

**Dataset sweep (2026-07-07, FINDINGS §8).** Winners, all HuggingFace, streaming, permissive:
- `HuggingFaceFW/fineweb-edu` (`sample-10BT`) — English, quality-filtered web. **Default.**
- `HuggingFaceFW/fineweb-2` (per language-script, e.g. `arb_Arab`, `zho_Hans`) — 1000+ languages.
- `bigcode/the-stack-smol` — source code, 600+ languages (content-bearing).

A byte-level tokenizer wins most on multilingual/code, so the default recipe **mixes** English +
code + one non-Latin script. Runs on a free Colab CPU/GPU; ~20 MB of text is plenty for a tokenizer.

In [ ]:
# === SETUP: install streaming datasets lib + get the Ada code ===
!pip -q install datasets
import os, sys
if not os.path.exists('best-tokenizer'):
    !git clone -q https://github.com/genmnz/best-tokenizer
sys.path.insert(0, 'best-tokenizer/lab/ada')
print('ready')

In [ ]:
# === DATA CONFIG (rule 10) === pick your corpus here ===
from ada_data import DataConfig, build_corpus
data_cfg = DataConfig(
    source='fineweb-edu',        # or 'synthetic' (offline), 'fineweb-2', 'code'
    mix=['fineweb-edu', 'code', 'fineweb-2'],  # [] = single source; recipe = English+code+multilingual
    config_name=None,            # for fineweb-2 set a language-script, e.g. 'arb_Arab'
    max_chars=20_000_000,        # ~20 MB total across the mix
    seed=0,
)
lines = build_corpus(data_cfg)
k = int(len(lines) * 0.9); train, test = lines[:k], lines[k:]
tot = sum(len(l.encode()) for l in train)
print(f'corpus: {len(lines):,} lines, train={len(train):,} test={len(test):,}, train bytes={tot/1e6:.1f} MB')

In [ ]:
# === TRAIN Ada v0 on REAL text + ablation (the open question from WORKPLAN Phase 3) ===
from dataclasses import replace
from ada_tokenizer import AdaTokenizer, AdaConfig
base = AdaConfig(vocab_size=8192)   # bump vocab for real text
configs = {
  'bpe-baseline':       replace(base, enable_superword=False, enable_entropy_guard=False),
  '+superword':         replace(base, enable_superword=True,  enable_entropy_guard=False),
  '+superword+entropy': replace(base, enable_superword=True,  enable_entropy_guard=True),
}
test_bytes = sum(len(t.encode()) for t in test)
for name, c in configs.items():
    tok = AdaTokenizer(c).train(train)
    toks = sum(len(tok.encode(t)) for t in test)
    print(f'{name:22s} bytes/tok = {test_bytes/toks:6.3f}  (vocab {tok.vocab_size})')
# Save the winner for reuse
best = AdaTokenizer(configs['+superword+entropy']).train(train); best.save('ada_v0.json')
print('saved ada_v0.json')

## Ada v1 — neural front-end (optional, needs a GPU runtime)
Learns boundaries end-to-end instead of from corpus stats. Must beat v0's bytes/tok to ship.

In [ ]:
# === V1 CONFIG (rule 10) ===
from dataclasses import dataclass, asdict
@dataclass
class V1Config:
    d_model:int=128; n_enc_layers:int=2; n_main_layers:int=4; n_heads:int=4
    fsq_levels:tuple=(8,8,8,5,5,5)        # implicit codebook = prod = 64000
    target_compression:float=0.5; delete_reg_weight:float=0.1; ratio_loss_weight:float=0.03
    seq_len:int=512; batch_size:int=32; lr:float=3e-4; steps:int=2000; seed:int=0
    run_name:str='ada_v1_r1'
vc=V1Config(); print(asdict(vc))

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, random
torch.manual_seed(vc.seed); dev='cuda' if torch.cuda.is_available() else 'cpu'
def round_ste(z): return z + (torch.round(z)-z).detach()
class FSQ(nn.Module):  # arXiv:2309.15505 (Cosmos best/cosmos-tokenizer quantizers.py)
    def __init__(s,levels): super().__init__(); s.register_buffer('L',torch.tensor(levels))
    def forward(s,z): half=(s.L-1)*0.5; return round_ste(torch.tanh(z)*half)/half
class Router(nn.Module):  # H-Net best/hnet/hnet/modules/dc.py:69
    def __init__(s,d):
        super().__init__(); s.q=nn.Linear(d,d,bias=False); s.k=nn.Linear(d,d,bias=False)
        with torch.no_grad(): s.q.weight.copy_(torch.eye(d)); s.k.weight.copy_(torch.eye(d))
    def forward(s,h):
        cos=F.cosine_similarity(s.q(h[:,:-1]),s.k(h[:,1:]),dim=-1)
        return F.pad(((1-cos)/2).clamp(0,1),(1,0),value=1.0)
class DeleteGate(nn.Module):  # MrT5 ScaledSigmoid best/mrt5/models/modeling_mrt5.py:104
    def __init__(s,d,scale=-30.0):
        super().__init__(); s.ff=nn.Sequential(nn.Linear(d,d),nn.GELU(),nn.Linear(d,1)); s.scale=scale
    def forward(s,h): return s.scale*torch.sigmoid(-s.ff(h).squeeze(-1))
class AdaV1(nn.Module):
    def __init__(s,c):
        super().__init__(); d=c.d_model; s.emb=nn.Embedding(320,d)
        s.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(d,c.n_heads,4*d,batch_first=True,activation='gelu'),c.n_enc_layers)
        s.router=Router(d); s.gate=DeleteGate(d)
        s.pin=nn.Linear(d,len(c.fsq_levels)); s.fsq=FSQ(c.fsq_levels); s.pout=nn.Linear(len(c.fsq_levels),d)
        s.main=nn.TransformerEncoder(nn.TransformerEncoderLayer(d,c.n_heads,4*d,batch_first=True,activation='gelu'),c.n_main_layers)
        s.head=nn.Linear(d,320)
    def forward(s,x):
        h=s.encoder(s.emb(x)); bprob=s.router(h); gate=s.gate(h)
        z=s.pout(s.fsq(s.pin(h))); h2=s.main(z+h); return s.head(h2),bprob,gate

In [ ]:
# Byte batches drawn from the REAL streamed corpus (train split from the data cell).
_blob = ('\n'.join(train)).encode('utf-8', 'replace')
rng=random.Random(vc.seed)
def byte_batch(c):
    rows=[]
    for _ in range(c.batch_size):
        i=rng.randint(0, max(1,len(_blob)-c.seq_len-1)); rows.append(list(_blob[i:i+c.seq_len]))
    return torch.tensor(rows, device=dev)

In [ ]:
# Train v1. Loss = next-byte CE + MrT5 deletion-rate reg + H-Net boundary-ratio balance.
model=AdaV1(vc).to(dev); opt=torch.optim.AdamW(model.parameters(), lr=vc.lr); log=[]
for step in range(vc.steps):
    x=byte_batch(vc); logits,bprob,gate=model(x)
    ce=F.cross_entropy(logits[:,:-1].reshape(-1,320), x[:,1:].reshape(-1))
    keep_rate=(gate > -1.0).float().mean()
    del_reg=vc.delete_reg_weight*((keep_rate-vc.target_compression)**2)
    ratio=vc.ratio_loss_weight*((bprob.mean()-vc.target_compression)**2)
    loss=ce+del_reg+ratio; opt.zero_grad(); loss.backward(); opt.step()
    if step%200==0:
        line=f'step {step} ce {ce.item():.3f} bpb {ce.item()/math.log(2):.3f} keep {keep_rate.item():.2f}'
        print(line); log.append(line)

In [ ]:
# Save + download two artifacts per run (rule 12): checkpoint zip + run report.
import json, zipfile
torch.save(model.state_dict(), f'{vc.run_name}.pt')
open(f'{vc.run_name}_report.md','w').write(
    f'# {vc.run_name}\n\ndata: {asdict(data_cfg)}\n\nv1cfg: {json.dumps(asdict(vc))}\n\n'+'\n'.join(log))
with zipfile.ZipFile(f'{vc.run_name}.zip','w') as z: z.write(f'{vc.run_name}.pt'); z.write('ada_v0.json')
try:
    from google.colab import files; files.download(f'{vc.run_name}.zip'); files.download(f'{vc.run_name}_report.md')
except Exception as e: print('not on colab:', e)